# Kazakhstan Government Budget — EDA

Exploratory analysis of the [Kazakhstan Gov Budget dataset](https://www.kaggle.com/datasets/baidalinadilzhan/kazakhstan-gov-budget-data) (`budget.csv`, converted from the original `budget.parquet`).

**Columns:** `id`, `year`, `budget_type`, `revenue`, `expenditure`, `budget_balance`, `revenue_category`, `expenditure_category`, `region`, `agency`, `budget_planned`, `budget_executed`, `execution_rate`, `program_code`, `program_name`, `subProgram_code`, `subProgram_name`, `amount`.

## Imports

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
%matplotlib inline

## Load the data

In [6]:
df = pd.read_csv('budget.csv')
df.head()

,id,year,budget_type,revenue,expenditure,budget_balance,revenue_category,expenditure_category,region,agency,budget_planned,budget_executed,execution_rate,program_code,program_name,subProgram_code,subProgram_name,amount
0,1,2024,республиканский,18500000000000,2.210000e+13,-3.600000e+12,налоги,образование,республиканский уровень,Министерство образования и науки РК,2800000000000,2650000000000,94.6,1,Развитие образования,001-1,Дошкольное и среднее образование,1200000000000
1,2,2024,республиканский,18500000000000,2.210000e+13,-3.600000e+12,налоги,здравоохранение,республиканский уровень,Министерство здравоохранения РК,2200000000000,2090000000000,95.0,2,Развитие здравоохранения,002-1,Первичная медико-санитарная помощь,950000000000
2,3,2024,местный,7200000000000,7.450000e+12,-2.500000e+11,налоги,социальная помощь,Алматы,Акимат г. Алматы,850000000000,820000000000,96.5,3,Социальная поддержка населения,003-1,Адресная социальная помощь,420000000000
3,4,2024,республиканский,18500000000000,2.210000e+13,-3.600000e+12,неналоговые,оборона,республиканский уровень,Министерство обороны РК,1800000000000,1750000000000,97.2,4,Обеспечение обороноспособности,004-1,Содержание вооруженных сил,1200000000000
4,5,2024,местный,4800000000000,4.950000e+12,-1.500000e+11,налоги,транспорт,Астана,Акимат г. Астана,650000000000,635000000000,97.7,5,Развитие транспортной инфраструктуры,005-1,Дорожное строительство,380000000000


## Make the big money columns readable (convert to billions)

The monetary columns are in tenge and run into the trillions, so they're hard to read. Convert them to **billions** (divide by 1e9) so the numbers are smaller. We do this once here, before any analysis, so every question below uses the readable values.

> Note: `execution_rate` is a percentage and `*_code`/`year`/`id` are identifiers — we leave those alone.

In [7]:
money_cols = ['revenue', 'expenditure', 'budget_balance',
              'budget_planned', 'budget_executed', 'amount']

# convert to billions of tenge (1e9) and round for readability
df[money_cols] = (df[money_cols] / 1e9).round(2)

# rename so the unit is clear in the column headers
df = df.rename(columns={c: c + '_bln' for c in money_cols})

df.head()

,id,year,budget_type,revenue_bln,expenditure_bln,budget_balance_bln,revenue_category,expenditure_category,region,agency,budget_planned_bln,budget_executed_bln,execution_rate,program_code,program_name,subProgram_code,subProgram_name,amount_bln
0,1,2024,республиканский,18500.0,22100.0,-3600.0,налоги,образование,республиканский уровень,Министерство образования и науки РК,2800.0,2650.0,94.6,1,Развитие образования,001-1,Дошкольное и среднее образование,1200.0
1,2,2024,республиканский,18500.0,22100.0,-3600.0,налоги,здравоохранение,республиканский уровень,Министерство здравоохранения РК,2200.0,2090.0,95.0,2,Развитие здравоохранения,002-1,Первичная медико-санитарная помощь,950.0
2,3,2024,местный,7200.0,7450.0,-250.0,налоги,социальная помощь,Алматы,Акимат г. Алматы,850.0,820.0,96.5,3,Социальная поддержка населения,003-1,Адресная социальная помощь,420.0
3,4,2024,республиканский,18500.0,22100.0,-3600.0,неналоговые,оборона,республиканский уровень,Министерство обороны РК,1800.0,1750.0,97.2,4,Обеспечение обороноспособности,004-1,Содержание вооруженных сил,1200.0
4,5,2024,местный,4800.0,4950.0,-150.0,налоги,транспорт,Астана,Акимат г. Астана,650.0,635.0,97.7,5,Развитие транспортной инфраструктуры,005-1,Дорожное строительство,380.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 615 entries, 0 to 614
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    615 non-null    int64  
 1   year                  615 non-null    int64  
 2   budget_type           615 non-null    object 
 3   revenue_bln           615 non-null    float64
 4   expenditure_bln       615 non-null    float64
 5   budget_balance_bln    615 non-null    float64
 6   revenue_category      615 non-null    object 
 7   expenditure_category  615 non-null    object 
 8   region                615 non-null    object 
 9   agency                615 non-null    object 
 10  budget_planned_bln    615 non-null    float64
 11  budget_executed_bln   615 non-null    float64
 12  execution_rate        615 non-null    float64
 13  program_code          615 non-null    int64  
 14  program_name          615 non-null    object 
 15  subProgram_code       6

In [ ]:
df.describe()

## Basic questions

**Which years are covered, and how many records per year?**

In [9]:
df['year'].value_counts().sort_index()

year
2022      2
2023     10
2024    411
2025    182
2026      5
2027      5
Name: count, dtype: int64

**What are the budget types (республиканский / местный) and their split?**

In [10]:
df['budget_type'].value_counts()

budget_type
республиканский    324
местный            291
Name: count, dtype: int64

**Total `amount` (billions) by expenditure category (top 10).**

In [11]:
df.groupby('expenditure_category')['amount_bln'].sum().sort_values(ascending=False).head(10)

expenditure_category
образование              11832.04
здравоохранение          10105.97
оборона                   9236.55
транспорт                 7600.54
социальная помощь         6848.46
сельское хозяйство        1888.25
внутренние дела           1705.40
индустрия                 1535.00
экономические вопросы     1420.00
государственный долг      1405.00
Name: amount_bln, dtype: float64

**Total `amount` (billions) by region (top 10).**

In [12]:
df.groupby('region')['amount_bln'].sum().sort_values(ascending=False).head(10)

region
республиканский уровень           59922.01
Алматы                             1880.00
Астана                             1785.00
Шымкент                            1240.00
Карагандинская область              799.22
Восточно-Казахстанская область      428.07
Западно-Казахстанская область       378.67
Южно-Казахстанская область          323.28
Северо-Казахстанская область        300.48
Костанайская область                267.92
Name: amount_bln, dtype: float64